# Data Cleaning: Stock Movement Context Dashboard
**Student:** Shreya · **Branch:** `cleaning-shreya` · **Data:** [TheFinAI/Herculean](https://huggingface.co/datasets/TheFinAI/Herculean), commit `52d3ecb`

Rules, reasons and checks are in [`docs/cleaning_decision_log.md`](docs/cleaning_decision_log.md).

## Phase 1: Preserve and inspect (read-only)
Nothing in this section modifies data. Raw files in `data/raw/` are only read.

In [1]:
import hashlib
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

RAW = Path("data/raw")
TABLES = ["prices", "news", "filings"]

### 1.1 Preserve the source (checklist #3)
SHA-256 of each raw file must equal the hash Hugging Face publishes for commit `52d3ecb97e490a2392717935402cde283e8fa36e`. A match proves the files are unmodified.

In [2]:
EXPECTED_SHA256 = {
    "prices":  "736c124db22e976dbf4ec1dd65e5a983497e5890e7999f7925b343dee5c965d5",
    "news":    "1fd109796eada02af37a15178cc150ec252e7c02bc440c3f76641eb8fab6da25",
    "filings": "822a854865651e042c9d64288785efa0f2c0fecdec2a16016ff3c2e7a5601299",
}
for t in TABLES:
    digest = hashlib.sha256((RAW / f"{t}.parquet").read_bytes()).hexdigest()
    print(f"{t:8s} {digest}  match={digest == EXPECTED_SHA256[t]}")
    assert digest == EXPECTED_SHA256[t], f"{t} raw file has changed"


prices   736c124db22e976dbf4ec1dd65e5a983497e5890e7999f7925b343dee5c965d5  match=True
news     1fd109796eada02af37a15178cc150ec252e7c02bc440c3f76641eb8fab6da25  match=True
filings  822a854865651e042c9d64288785efa0f2c0fecdec2a16016ff3c2e7a5601299  match=True


In [3]:
raw = {t: pd.read_parquet(RAW / f"{t}.parquet") for t in TABLES}
prices, news, filings = raw["prices"], raw["news"], raw["filings"]
pd.DataFrame({t: {"rows": len(df), "cols": df.shape[1]} for t, df in raw.items()})

,prices,news,filings
rows,2656,3888,73
cols,9,4,6


### 1.2 Columns and types (checklist #4)
Parquet stores `date` as a true date (`date32`), but pandas loads it as `object` (Python `date`). That is the only type to convert (rule R1).

In [4]:
pd.DataFrame({t: df.dtypes.astype(str) for t, df in raw.items()}).fillna("")

,prices,news,filings
adj_close,float64,,
close,float64,,
date,object,object,object
document_type,,,str
high,float64,,
highlights,,str,
id,int32,int32,int32
low,float64,,
mda_content,,,str
open,float64,,


In [5]:
prices.head(3)

,id,symbol,date,open,high,low,close,adj_close,volume
0,1,AAPL,2024-12-02,237.270004,240.789993,237.160004,239.589996,238.288300,48137100
1,2,AAPL,2024-12-03,239.809998,242.759995,238.899994,242.649994,241.331665,38861000
2,3,AAPL,2024-12-04,242.869995,244.110001,241.250000,243.009995,241.689728,44383900


In [6]:
# Long text columns shown as lengths only
filings.assign(mda_len=filings.mda_content.str.len(), risk_len=filings.risk_content.str.len())[
    ["id", "symbol", "date", "document_type", "mda_len", "risk_len"]].head(3)

,id,symbol,date,document_type,mda_len,risk_len
0,1,AAPL,2024-02-01,10-Q,15930,4592
1,2,AAPL,2024-05-02,10-Q,17490,4804
2,3,AAPL,2024-08-01,10-Q,18125,986


### 1.3 Missing values (checklist #7, before)

In [7]:
pd.concat({t: df.isna().sum() for t, df in raw.items()}).rename("nulls").to_frame().T

prices                                                  news                        filings                                                   
          id symbol date open high low close adj_close volume   id symbol date highlights      id symbol date document_type mda_content risk_content
nulls      0      0    0    0    0   0     0         0      0    0      0    0          0       0      0    0             0           0            0

### 1.4 One row and unique key (checklists #2 and #6)

In [8]:
KEYS = {"prices": ["symbol", "date"], "news": ["symbol", "date"], "filings": ["symbol", "date", "document_type"]}
pd.DataFrame({
    t: {
        "rows": len(df),
        "exact_dup_rows": df.duplicated().sum(),
        "dup_rows_ignoring_id": df.drop(columns="id").duplicated().sum(),
        "repeated_id": df.id.duplicated().sum(),
        f"repeated_key": df.duplicated(KEYS[t]).sum(),
        "symbols": df.symbol.nunique(),
        "min_date": df.date.min(),
        "max_date": df.date.max(),
    } for t, df in raw.items()
})

,prices,news,filings
rows,2656,3888,73
exact_dup_rows,0,0,0
dup_rows_ignoring_id,0,0,0
repeated_id,0,0,0
repeated_key,0,0,0
symbols,8,8,8
min_date,2024-12-02,2024-12-01,2024-01-16
max_date,2026-03-31,2026-03-31,2026-03-25


In [9]:
# prices: every symbol has the same 332 trading days
print("rows per symbol:", prices.groupby("symbol").size().unique())
print("identical date set across symbols:", prices.groupby("symbol").date.apply(frozenset).nunique() == 1)

# news: one row for every calendar day for every symbol
calendar = set(pd.date_range(news.date.min(), news.date.max()).date)
print("calendar days:", len(calendar), "| missing per symbol:",
      {s: len(calendar - set(g.date)) for s, g in news.groupby("symbol")})

# every trading day has a news row
m = prices[["symbol", "date"]].merge(news[["symbol", "date"]], how="left", indicator=True)
print("trading days without news:", (m._merge == "left_only").sum())
print("weekend news rows:", (pd.to_datetime(news.date).dt.dayofweek >= 5).sum())

rows per symbol: [332]
identical date set across symbols: True
calendar days: 486 | missing per symbol: {'AAPL': 0, 'ADBE': 0, 'AMZN': 0, 'GOOGL': 0, 'META': 0, 'MSFT': 0, 'NVDA': 0, 'TSLA': 0}
trading days without news: 0
weekend news rows: 1112


### 1.5 Suspicious values (checklist #8, inspection only)

In [10]:
print({
    "low > high": (prices.low > prices.high).sum(),
    "open outside [low, high]": ((prices.open < prices.low) | (prices.open > prices.high)).sum(),
    "close outside [low, high]": ((prices.close < prices.low) | (prices.close > prices.high)).sum(),
    "non-positive price": (prices[["open", "high", "low", "close", "adj_close"]] <= 0).any(axis=1).sum(),
    "volume <= 0": (prices.volume <= 0).sum(),
})
ratio = prices.adj_close / prices.close
print("adj_close/close jumps > 2% (splits):", (ratio.groupby(prices.symbol).pct_change().abs() > 0.02).sum())
closures = ["2024-12-25", "2025-01-09", "2025-07-04", "2025-11-27", "2026-01-01"]
print("market closures present in prices:", {d: pd.Timestamp(d).date() in set(prices.date) for d in closures})

{'low > high': np.int64(0), 'open outside [low, high]': np.int64(0), 'close outside [low, high]': np.int64(0), 'non-positive price': np.int64(0), 'volume <= 0': np.int64(0)}


adj_close/close jumps > 2% (splits): 0
market closures present in prices: {'2024-12-25': False, '2025-01-09': False, '2025-07-04': False, '2025-11-27': False, '2026-01-01': False}


In [11]:
# Extreme daily ranges: inspected, not removed (unusual is not wrong)
rng = ((prices.high - prices.low) / prices.close).rename("daily_range_pct")
prices.assign(daily_range_pct=rng).nlargest(6, "daily_range_pct")[
    ["id", "symbol", "date", "high", "low", "close", "daily_range_pct"]]

,id,symbol,date,high,low,close,daily_range_pct
2411,2412,TSLA,2025-04-09,274.690002,223.880005,272.200012,0.186664
2450,2451,TSLA,2025-06-05,324.549988,273.209991,284.700012,0.180330
2409,2410,TSLA,2025-04-07,252.000000,214.250000,233.289993,0.161816
2077,2078,NVDA,2025-04-07,101.750000,86.620003,97.639999,0.154957
2079,2080,NVDA,2025-04-09,115.099998,97.529999,114.330002,0.153678
2389,2390,TSLA,2025-03-10,253.369995,220.000000,222.149994,0.150214


### 1.6 Text quality (checklist #5)

In [12]:
def text_profile(s):
    return {
        "empty": (s.str.strip() == "").sum(),
        "outer_whitespace": (s != s.str.strip()).sum(),
        "html_entities": s.str.contains(r"&#\d+;", regex=True).sum(),
        "len_min": s.str.len().min(),
        "len_median": int(s.str.len().median()),
        "duplicate_texts": s.duplicated().sum(),
    }

pd.DataFrame({
    "news.highlights": text_profile(news.highlights),
    "filings.mda_content": text_profile(filings.mda_content),
    "filings.risk_content": text_profile(filings.risk_content),
})

,news.highlights,filings.mda_content,filings.risk_content
empty,0,0,0
outer_whitespace,0,73,73
html_entities,0,73,68
len_min,25,15930,333
len_median,4719,46697,66162
duplicate_texts,0,0,7


In [13]:
print("symbols:", sorted(prices.symbol.unique()))
print("document_type:", filings.document_type.value_counts().to_dict())
print("\nmda_content raw start (id 1):", repr(filings.loc[filings.id == 1, "mda_content"].iloc[0][:45]))

symbols: ['AAPL', 'ADBE', 'AMZN', 'GOOGL', 'META', 'MSFT', 'NVDA', 'TSLA']
document_type: {'10-Q': 51, '10-K': 22}

mda_content raw start (id 1): ' Item 2. Management&#8217;s Discussion and An'


In [14]:
# Low-information LLM summaries (< 300 chars): to be flagged, not deleted (rule R5)
short = news[news.highlights.str.len() < 300]
short.assign(chars=short.highlights.str.len(), preview=short.highlights.str[:90])[["id", "symbol", "date", "chars", "preview"]]

,id,symbol,date,chars,preview
542,543,ADBE,2025-01-26,25,The article also includes
1004,1005,AMZN,2025-01-02,153,"This item conveys a clear negative signal for that specific date, indicating..."
1488,1489,GOOGL,2024-12-31,138,"Consequently, there are no substantive GOOGL events, themes, or market senti..."
1957,1958,META,2024-12-14,222,"The sole\n\nAs a result, no themes, developments, or market sentiment relate..."
1961,1962,META,2024-12-18,288,"Because no META content is present in the supplied material for that date, t..."


In [15]:
# 10-Q risk sections that only refer back to the 10-K (rule R6)
short_risk = filings[filings.risk_content.str.len() < 2000]
short_risk.assign(chars=short_risk.risk_content.str.len())[["id", "symbol", "date", "document_type", "chars"]]

,id,symbol,date,document_type,chars
2,3,AAPL,2024-08-01,10-Q,986
6,7,AAPL,2025-08-01,10-Q,1060
8,9,AAPL,2026-01-30,10-Q,440
33,34,GOOGL,2025-04-24,10-Q,600
34,35,GOOGL,2025-07-23,10-Q,600
35,36,GOOGL,2025-10-29,10-Q,600
65,66,TSLA,2024-04-23,10-Q,333
66,67,TSLA,2024-07-23,10-Q,333
67,68,TSLA,2024-10-23,10-Q,333
71,72,TSLA,2025-10-22,10-Q,613


### 1.7 What does `filings.date` mean?
Filing date vs period end. AAPL's fiscal year ends in late September and MSFT's on June 30, yet their 10-Ks are dated about a month later, so `date` is the **filing date**. An as-of join on it has no look-ahead.

In [16]:
filings[filings.document_type == "10-K"].query("symbol in ['AAPL', 'MSFT', 'NVDA']")[["id", "symbol", "date", "document_type"]]

,id,symbol,date,document_type
3,4,AAPL,2024-11-01,10-K
7,8,AAPL,2025-10-31,10-K
48,49,MSFT,2024-07-30,10-K
52,53,MSFT,2025-07-30,10-K
55,56,NVDA,2024-02-21,10-K
59,60,NVDA,2025-02-26,10-K
63,64,NVDA,2026-02-25,10-K


In [17]:
print("filings dated before first trading day:", (filings.date < prices.date.min()).sum())

filings dated before first trading day: 32


### Phase 1 summary
- Raw files verified byte-for-byte against HF commit `52d3ecb`.
- 0 nulls, 0 duplicates, 0 repeated keys, complete date grids, valid OHLC.
- Issues for Phase 2: `date` dtype (R1), HTML entities and whitespace in filing text (R4), 5 broken news summaries (R5), 10 reference-only 10-Q risk sections (R6).
- Five record checks selected with expected results written in advance. See `docs/cleaning_decision_log.md` §5.

## Phase 2: Clean
All rules are in `src/dso576_python/cleaning.py`. The notebook calls it, so the notebook and `uv run python src/dso576_python/cleaning.py` produce identical outputs. The raw files are re-verified before cleaning.

In [18]:
import sys
sys.path.insert(0, "src")
from dso576_python import cleaning

cleaning.main()
changes = pd.read_csv("outputs/cleaning_changes.csv", dtype={"nulls_before": "Int64"})
changes

wrote data/cleaned/prices.parquet  (2,656 rows)


wrote data/cleaned/news.parquet  (3,888 rows)
wrote data/cleaned/filings.parquet  (73 rows)


wrote data/cleaned/price_context.parquet  (2,656 rows)
wrote outputs/cleaning_changes.csv
wrote sample/price_context_sample.csv  (40 rows)
wrote sample/news_sample.csv  (5 rows)
wrote sample/filings_sample.csv  (5 rows)


,rule,table,column,action,rows_before,rows_after,nulls_before,nulls_after,values_changed
0,R1,prices,date,object -> datetime64,2656,2656,0,0,2656
1,R2,prices,symbol,strip + upper; validate 8 tickers,2656,2656,0,0,0
2,R8,prices,daily_range_pct,derived (high - low) / close; no rows removed,2656,2656,<NA>,0,0
3,R1,news,date,object -> datetime64,3888,3888,0,0,3888
4,R2,news,symbol,strip + upper; validate 8 tickers,3888,3888,0,0,0
5,R5,news,highlights_low_info,flag len(highlights) < 300,3888,3888,<NA>,0,5
6,R1,filings,date,object -> datetime64,73,73,0,0,73
7,R2,filings,symbol,strip + upper; validate 8 tickers,73,73,0,0,0
8,R3,filings,document_type,validate 10-K or 10-Q only,73,73,0,0,0
9,R4,filings,mda_content,html.unescape + strip,73,73,0,0,73


In [19]:
clean = {t: pd.read_parquet(f"data/cleaned/{t}.parquet") for t in TABLES}
c_prices, c_news, c_filings = clean["prices"], clean["news"], clean["filings"]
pd.DataFrame({t: clean[t].dtypes.astype(str) for t in TABLES}).fillna("")

,prices,news,filings
adj_close,float64,,
close,float64,,
daily_range_pct,float64,,
date,datetime64[ms],datetime64[ms],datetime64[ms]
document_type,,,str
high,float64,,
highlights,,str,
highlights_low_info,,bool,
id,int32,int32,int32
low,float64,,


### 2.1 Types (checklist #4): `date` object → datetime64
Failed conversions = values that became NaT. The script raises an error if any occur.

In [20]:
pd.DataFrame({
    t: {
        "dtype_before": str(raw[t].date.dtype),
        "dtype_after": str(clean[t].date.dtype),
        "failed_conversions": int(clean[t].date.isna().sum()),
        "min_unchanged": pd.Timestamp(raw[t].date.min()) == clean[t].date.min(),
        "max_unchanged": pd.Timestamp(raw[t].date.max()) == clean[t].date.max(),
    } for t in TABLES
})

,prices,news,filings
dtype_before,object,object,object
dtype_after,datetime64[ms],datetime64[ms],datetime64[ms]
failed_conversions,0,0,0
min_unchanged,True,True,True
max_unchanged,True,True,True


### 2.2 Text before and after (checklist #5)
Filing text: HTML entities decoded and outer whitespace stripped. Wording, case, and line breaks are untouched. News highlights and symbols needed no change.

In [21]:
before = filings.loc[filings.id == 1, "mda_content"].iloc[0]
after = c_filings.loc[c_filings.id == 1, "mda_content"].iloc[0]
print("BEFORE:", repr(before[:60]))
print("AFTER: ", repr(after[:60]))
print("\nentity rows before/after:",
      {c: (int(filings[c].str.contains(r"&#\d+;").sum()), int(c_filings[c].str.contains(r"&#\d+;").sum()))
       for c in ["mda_content", "risk_content"]})
print("outer-whitespace rows before/after:",
      {c: (int((filings[c] != filings[c].str.strip()).sum()), int((c_filings[c] != c_filings[c].str.strip()).sum()))
       for c in ["mda_content", "risk_content"]})
print("news highlights changed:", int((c_news.highlights != news.highlights).sum()))
print("symbols changed:", {t: int((clean[t].symbol != raw[t].symbol).sum()) for t in TABLES})

BEFORE: ' Item 2. Management&#8217;s Discussion and Analysis of Finan'
AFTER:  'Item 2. Management’s Discussion and Analysis of Financial Co'

entity rows before/after: {'mda_content': (73, 0), 'risk_content': (68, 0)}
outer-whitespace rows before/after: {'mda_content': (73, 0), 'risk_content': (73, 0)}
news highlights changed: 0
symbols changed: {'prices': 0, 'news': 0, 'filings': 0}


In [22]:
# Only entities and outer whitespace changed: the text with entities decoded is identical apart from the strip
import html
for c in ["mda_content", "risk_content"]:
    same = (filings[c].map(html.unescape).str.strip() == c_filings[c]).all()
    print(f"{c}: cleaned == unescape(raw).strip() for all rows -> {same}")

mda_content: cleaned == unescape(raw).strip() for all rows -> True
risk_content: cleaned == unescape(raw).strip() for all rows -> True


### 2.3 Duplicates (checklist #6)
Counted separately as exact duplicate rows, repeated `id`s, and repeated business keys. All are 0, so nothing was removed (R7). Repeated `risk_content` text (7 repeats across 11 rows) is copied disclosure language across quarters, not a duplicate record, so it is kept.

In [23]:
pd.DataFrame({
    t: {
        "exact_dup_rows": int(clean[t].duplicated().sum()),
        "repeated_id": int(clean[t].id.duplicated().sum()),
        "repeated_key": int(clean[t].duplicated(KEYS[t]).sum()),
        "rows_removed": len(raw[t]) - len(clean[t]),
    } for t in TABLES
})

,prices,news,filings
exact_dup_rows,0,0,0
repeated_id,0,0,0
repeated_key,0,0,0
rows_removed,0,0,0


In [24]:
dup_risk = c_filings[c_filings.risk_content.duplicated(keep=False)]
dup_risk.assign(chars=dup_risk.risk_content.str.len())[["id", "symbol", "date", "document_type", "chars"]]

,id,symbol,date,document_type,chars
21,22,AMZN,2024-08-01,10-Q,59365
22,23,AMZN,2024-10-31,10-Q,59365
24,25,AMZN,2025-05-01,10-Q,60089
25,26,AMZN,2025-07-31,10-Q,60089
26,27,AMZN,2025-10-30,10-Q,60089
33,34,GOOGL,2025-04-24,10-Q,591
34,35,GOOGL,2025-07-23,10-Q,591
35,36,GOOGL,2025-10-29,10-Q,591
65,66,TSLA,2024-04-23,10-Q,329
66,67,TSLA,2024-07-23,10-Q,329


### 2.4 Missing data (checklist #7)
No missing values before or after cleaning, so nothing was filled or dropped (R9). Low-quality text is **flagged** rather than set to missing, so the analyst still sees the original.

In [25]:
def null_counts(tables):
    return pd.concat({t: df.isna().sum() for t, df in tables.items()})

pd.DataFrame({"nulls_before": null_counts(raw), "nulls_after": null_counts(clean)}).astype("Int64")

nulls_before  nulls_after
filings date                            0            0
        document_type                   0            0
        id                              0            0
        mda_content                     0            0
        risk_content                    0            0
        risk_refers_to_10k           <NA>            0
        symbol                          0            0
news    date                            0            0
        highlights                      0            0
        highlights_low_info          <NA>            0
        id                              0            0
        symbol                          0            0
prices  adj_close                       0            0
        close                           0            0
        daily_range_pct              <NA>            0
        date                            0            0
        high                            0            0
        id                              0            0
        low                             0            0
        open                            0            0
        symbol                          0            0
        volume                          0            0

In [26]:
print("highlights_low_info (R5):", int(c_news.highlights_low_info.sum()))
print("risk_refers_to_10k (R6):", int(c_filings.risk_refers_to_10k.sum()),
      "| document types:", c_filings.loc[c_filings.risk_refers_to_10k, "document_type"].unique().tolist())
c_news.loc[c_news.highlights_low_info, ["id", "symbol", "date", "highlights"]]

highlights_low_info (R5): 5


risk_refers_to_10k (R6): 10 | document types: ['10-Q']


,id,symbol,date,highlights
542,543,ADBE,2025-01-26,The article also includes
1004,1005,AMZN,2025-01-02,"This item conveys a clear negative signal for that specific date, indicating..."
1488,1489,GOOGL,2024-12-31,"Consequently, there are no substantive GOOGL events, themes, or market senti..."
1957,1958,META,2024-12-14,"The sole\n\nAs a result, no themes, developments, or market sentiment relate..."
1961,1962,META,2024-12-18,"Because no META content is present in the supplied material for that date, t..."


### Phase 2 summary
- Rows in = rows out: prices 2,656 · news 3,888 · filings 73. No rows dropped, no values imputed.
- Changed values: `date` (all rows, type only), `mda_content` (73), `risk_content` (73). Everything else is unchanged.
- New columns: `daily_range_pct` (prices), `highlights_low_info` (news, 5 True), `risk_refers_to_10k` (filings, 10 True).
- Change log written to `outputs/cleaning_changes.csv`.

## Phase 3: Validate
`cleaning.main()` (run in Phase 2) also builds `data/cleaned/price_context.parquet`: one row per trading day with its flag, same-day news id, latest filing, and latest 10-K.

In [27]:
ctx = pd.read_parquet("data/cleaned/price_context.parquet")
c_filings = pd.read_parquet("data/cleaned/filings.parquet")  # reload: cleaning.main() rewrote it
print(ctx.shape)
ctx.head(3)

(2656, 18)


,price_id,symbol,date,high,low,close,daily_range_pct,is_flagged,co_flag_count,news_id,highlights_low_info,filing_id,filing_date,document_type,risk_refers_to_10k,filed_same_day,latest_10k_id,latest_10k_date
0,1,AAPL,2024-12-02,240.789993,237.160004,239.589996,0.015151,False,0,2,False,4,2024-11-01,10-K,False,False,4,2024-11-01
1,2,AAPL,2024-12-03,242.759995,238.899994,242.649994,0.015908,False,0,3,False,4,2024-11-01,10-K,False,False,4,2024-11-01
2,3,AAPL,2024-12-04,244.110001,241.250000,243.009995,0.011769,False,0,4,False,4,2024-11-01,10-K,False,False,4,2024-11-01


### 3.1 Suspicious values (checklist #8)
Invalid dates: prices on weekends, dates outside the documented ranges. Impossible values: inconsistent OHLC, non-positive prices or volume. Unusual records: extreme ranges are **kept and flagged**, because they are what the dashboard looks for.

In [28]:
print({
    "price rows on weekends": int((c_prices.date.dt.dayofweek >= 5).sum()),
    "prices outside 2024-12-02..2026-03-31": int((~c_prices.date.between("2024-12-02", "2026-03-31")).sum()),
    "news outside 2024-12-01..2026-03-31": int((~c_news.date.between("2024-12-01", "2026-03-31")).sum()),
    "filings after last price date": int((c_filings.date > c_prices.date.max()).sum()),
    "inconsistent OHLC": int(((c_prices.low > c_prices[["open", "close"]].min(axis=1)) |
                              (c_prices.high < c_prices[["open", "close"]].max(axis=1))).sum()),
    "non-positive price or volume": int(((c_prices[["open", "high", "low", "close"]] <= 0).any(axis=1) | (c_prices.volume <= 0)).sum()),
    "daily_range_pct <= 0 or > 0.5": int((~c_prices.daily_range_pct.between(0, 0.5, inclusive="right")).sum()),
})

{'price rows on weekends': 0, 'prices outside 2024-12-02..2026-03-31': 0, 'news outside 2024-12-01..2026-03-31': 0, 'filings after last price date': 0, 'inconsistent OHLC': 0, 'non-positive price or volume': 0, 'daily_range_pct <= 0 or > 0.5': 0}


In [29]:
# Unusual but real: days when 5+ of the 8 companies were flagged together (likely market-wide)
(ctx[ctx.is_flagged & (ctx.co_flag_count >= 5)]
   .groupby("date").agg(companies=("symbol", lambda s: ", ".join(s)), co_flag_count=("co_flag_count", "first")))

,companies,co_flag_count
date,,
2024-12-18,"AMZN, GOOGL, META, MSFT, TSLA",5
2025-04-04,"AAPL, AMZN, MSFT, NVDA, TSLA",5
2025-04-07,"AAPL, ADBE, AMZN, GOOGL, META, MSFT, NVDA, TSLA",8
2025-04-08,"AAPL, ADBE, AMZN, GOOGL, META, MSFT, NVDA, TSLA",8
2025-04-09,"AAPL, ADBE, AMZN, GOOGL, META, MSFT, NVDA, TSLA",8
2025-04-10,"AAPL, ADBE, AMZN, GOOGL, META, MSFT, NVDA, TSLA",8
2025-11-20,"AMZN, GOOGL, MSFT, NVDA, TSLA",5


### 3.2 Verify merges (checklist #9)
| Merge | Keys | Expected relationship |
|-------|------|-----------------------|
| prices → news | `symbol`, `date` | one-to-one (every trading day has exactly one news row; news has extra non-trading days) |
| prices → latest filing | `symbol`, as-of `date` (backward, same day allowed) | many-to-one (each trading day gets at most one filing) |
| prices → latest 10-K | `symbol`, as-of `date` (backward, same day allowed) | many-to-one |

`validate="one_to_one"` in the script makes pandas raise an error if the news keys are not unique.

In [30]:
m = c_prices[["symbol", "date"]].merge(c_news[["symbol", "date"]], on=["symbol", "date"], how="outer", indicator=True)
print("prices -> news:", m._merge.value_counts().to_dict())
print("  news rows not matched to a trading day (weekends/holidays, expected):", int((m._merge == "right_only").sum()),
      "=", len(c_news), "-", len(c_prices))

pd.DataFrame({
    "rows_before": [len(c_prices)] * 3,
    "rows_after": [len(ctx)] * 3,
    "unmatched_left": [int(ctx.news_id.isna().sum()), int(ctx.filing_id.isna().sum()), int(ctx.latest_10k_id.isna().sum())],
    "duplicate (symbol, date) after": [int(ctx.duplicated(["symbol", "date"]).sum())] * 3,
}, index=["news", "latest filing", "latest 10-K"])

prices -> news: {'both': 2656, 'right_only': 1232, 'left_only': 0}
  news rows not matched to a trading day (weekends/holidays, expected): 1232 = 3888 - 2656


,rows_before,rows_after,unmatched_left,"duplicate (symbol, date) after"
news,2656,2656,0,0
latest filing,2656,2656,0,0
latest 10-K,2656,2656,0,0


In [31]:
# Join correctness, checked independently of merge_asof
assert (ctx.filing_date <= ctx.date).all() and (ctx.latest_10k_date <= ctx.date).all()   # no look-ahead
assert ctx.symbol.eq(c_filings.set_index("id").loc[ctx.filing_id, "symbol"].values).all()  # same company
newer = ctx.merge(c_filings[["symbol", "date"]].rename(columns={"date": "fdate"}), on="symbol")
newer = newer[(newer.fdate > newer.filing_date) & (newer.fdate <= newer.date)]
print("cases where a more recent eligible filing was skipped:", len(newer))
print("same-day filings:", int(ctx.filed_same_day.sum()), "trading-day rows (= number of filings dated on a trading day in range)")
print("distinct filings used:", ctx.filing_id.nunique(), "| filings dated in price window:",
      int(c_filings.date.between(c_prices.date.min(), c_prices.date.max()).sum()))

cases where a more recent eligible filing was skipped: 0
same-day filings: 41 trading-day rows (= number of filings dated on a trading day in range)
distinct filings used: 49 | filings dated in price window: 41


### 3.3 Reconcile changes (checklist #10)

In [32]:
pd.DataFrame({
    "start_rows": {t: len(raw[t]) for t in TABLES} | {"price_context": len(prices)},
    "final_rows": {t: len(clean[t]) for t in TABLES} | {"price_context": len(ctx)},
}).assign(difference=lambda d: d.final_rows - d.start_rows)

,start_rows,final_rows,difference
prices,2656,2656,0
news,3888,3888,0
filings,73,73,0
price_context,2656,2656,0


In [33]:
# Important totals are unchanged by cleaning and merging
totals = pd.DataFrame({
    "raw": [prices.volume.sum(), prices.close.sum(), prices.groupby("symbol").date.nunique().sum(), filings.document_type.value_counts()["10-K"]],
    "cleaned": [c_prices.volume.sum(), c_prices.close.sum(), c_prices.groupby("symbol").date.nunique().sum(), c_filings.document_type.value_counts()["10-K"]],
    "price_context": [None, ctx.close.sum(), ctx.groupby("symbol").date.nunique().sum(), None],
}, index=["total volume", "sum of close", "symbol-trading days", "10-K filings"])
totals

,raw,cleaned,price_context
total volume,1.594568e+11,1.594568e+11,NaN
sum of close,8.947304e+05,8.947304e+05,894730.359695
symbol-trading days,2.656000e+03,2.656000e+03,2656.000000
10-K filings,2.200000e+01,2.200000e+01,NaN


### 3.4 Check calculations (checklist #11)
- `daily_range_pct = (high − low) / close`. The denominator is `close`, which is never missing or zero, so all 2,656 rows are used.
- Flag threshold = 95th percentile **per symbol** (332 days each). There are no missing values, so no row drops out of a percentile. Under R9, a row with null high, low or close would get NaN, and pandas `quantile` skips NaN, so the denominator would shrink. The record count per symbol is reported so that would be visible.
- Missing group keys: `symbol` has 0 nulls. `groupby` drops NaN keys by default, so a null symbol would silently disappear. The script raises an error on any symbol outside the 8 tickers instead.

In [34]:
calc = ctx.groupby("symbol").agg(
    rows_used=("daily_range_pct", "count"),
    threshold_p95=("daily_range_pct", lambda s: s.quantile(0.95)),
    flagged_days=("is_flagged", "sum"),
    mean_range=("daily_range_pct", "mean"),
)
print("null symbol keys:", int(ctx.symbol.isna().sum()), "| null close:", int(c_prices.close.isna().sum()),
      "| zero close:", int((c_prices.close == 0).sum()))
calc.round(4)

null symbol keys: 0 | null close: 0 | zero close: 0


,rows_used,threshold_p95,flagged_days,mean_range
symbol,,,,
AAPL,332,0.0433,17,0.0220
ADBE,332,0.0488,17,0.0254
AMZN,332,0.0455,17,0.0240
GOOGL,332,0.0445,17,0.0250
META,332,0.0472,17,0.0265
MSFT,332,0.0346,17,0.0184
NVDA,332,0.0671,17,0.0331
TSLA,332,0.0830,17,0.0442


In [35]:
print("total flagged days:", int(ctx.is_flagged.sum()), "of", len(ctx))
print("co_flag_count over flagged rows:", ctx[ctx.is_flagged].co_flag_count.value_counts().sort_index().to_dict())
print("flagged days with a news row:", int(ctx[ctx.is_flagged].news_id.notna().sum()),
      "| with low-info news:", int(ctx[ctx.is_flagged].highlights_low_info.sum()))
print("flagged days with a prior filing:", int(ctx[ctx.is_flagged].filing_id.notna().sum()))

total flagged days: 136 of 2656
co_flag_count over flagged rows: {1: 45, 2: 34, 3: 6, 4: 4, 5: 15, 8: 32}
flagged days with a news row: 136 | with low-info news: 1
flagged days with a prior filing: 136


### 3.5 Verify individual records (checklist #12)
Records and expected results were written in `docs/cleaning_decision_log.md` §5 **during Phase 1, before any cleaning code existed**. The actual values below come from the cleaned outputs.

In [36]:
def news_window(symbol, day, days=7):
    """News rows from day-7 through day: what the dashboard's news panel shows."""
    day = pd.Timestamp(day)
    return c_news[(c_news.symbol == symbol) & c_news.date.between(day - pd.Timedelta(days=days), day)]

def ctx_row(symbol, day):
    return ctx[(ctx.symbol == symbol) & (ctx.date == day)].iloc[0]

r1 = ctx_row("TSLA", "2025-04-09")
r2 = ctx_row("AAPL", "2024-12-02")
r3 = ctx_row("NVDA", "2025-02-26")
r4 = c_news[c_news.id == 543].iloc[0]
r5_raw = filings.loc[filings.id == 1, "mda_content"].iloc[0]
r5 = c_filings.loc[c_filings.id == 1, "mda_content"].iloc[0]

record_checks = pd.DataFrame([
    {"record": "prices id 2412 · TSLA 2025-04-09 (largest range)",
     "original": "high 274.690002, low 223.880005, close 272.200012",
     "expected": "daily_range_pct ≈ 0.18666, kept",
     "actual": f"daily_range_pct = {r1.daily_range_pct:.5f}, kept, is_flagged={r1.is_flagged}",
     "match": round(r1.daily_range_pct, 5) == 0.18666},
    {"record": "prices id 1 · AAPL 2024-12-02 (first trading day)",
     "original": "no filing inside the price window yet",
     "expected": "latest filing = id 4 (10-K, 2024-11-01)",
     "actual": f"latest filing = id {r2.filing_id} ({r2.document_type}, {r2.filing_date.date()})",
     "match": r2.filing_id == 4},
    {"record": "prices id 2050 · NVDA 2025-02-26 (filing same day)",
     "original": "NVDA 10-K id 60 dated 2025-02-26",
     "expected": "latest filing = id 60, filed_same_day = True",
     "actual": f"latest filing = id {r3.filing_id} ({r3.document_type}), filed_same_day = {r3.filed_same_day}",
     "match": (r3.filing_id == 60) and r3.filed_same_day},
    {"record": "news id 543 · ADBE 2025-01-26 (Sunday, broken summary)",
     "original": repr(news.loc[news.id == 543, "highlights"].iloc[0]),
     "expected": "text unchanged, low_info = True, in D−7 window of ADBE 2025-01-27",
     "actual": f"text {repr(r4.highlights)}, low_info = {r4.highlights_low_info}, "
               f"in window = {543 in news_window('ADBE', '2025-01-27').id.values}",
     "match": (r4.highlights == "The article also includes") and r4.highlights_low_info
              and (543 in news_window("ADBE", "2025-01-27").id.values)},
    {"record": "filings id 1 · AAPL 2024-02-01 10-Q (entity + space)",
     "original": repr(r5_raw[:32]),
     "expected": "'Item 2. Management’s Discussion…'",
     "actual": repr(r5[:31]),
     "match": r5.startswith("Item 2. Management\u2019s Discussion")},
])
with pd.option_context("display.max_colwidth", 70):
    display(record_checks)
assert record_checks.match.all()

,record,original,expected,actual,match
0,prices id 2412 · TSLA 2025-04-09 (largest range),"high 274.690002, low 223.880005, close 272.200012","daily_range_pct ≈ 0.18666, kept","daily_range_pct = 0.18666, kept, is_flagged=True",True
1,prices id 1 · AAPL 2024-12-02 (first trading day),no filing inside the price window yet,"latest filing = id 4 (10-K, 2024-11-01)","latest filing = id 4 (10-K, 2024-11-01)",True
2,prices id 2050 · NVDA 2025-02-26 (filing same day),NVDA 10-K id 60 dated 2025-02-26,"latest filing = id 60, filed_same_day = True","latest filing = id 60 (10-K), filed_same_day = True",True
3,"news id 543 · ADBE 2025-01-26 (Sunday, broken summary)",'The article also includes',"text unchanged, low_info = True, in D−7 window of ADBE 2025-01-27","text 'The article also includes', low_info = True, in window = True",True
4,filings id 1 · AAPL 2024-02-01 10-Q (entity + space),' Item 2. Management&#8217;s Disc','Item 2. Management’s Discussion…','Item 2. Management’s Discussion',True


### Phase 3 summary
- No invalid dates or impossible values. Extreme ranges are kept and flagged. 4 days had all 8 companies flagged together (likely market-wide).
- Merges: 2,656 rows in and 2,656 out, 0 unmatched, 0 duplicate keys, no look-ahead, no skipped newer filing.
- Totals (volume, sum of close, trading days, 10-K count) are unchanged by cleaning and merging.
- 136 flagged days (17 per symbol). All have news and a prior filing.
- All 5 record checks match their pre-written expectations.

## Phase 4: Outputs and reproducibility
### 4.1 Cleaned sample (50 rows)
Written by `cleaning.build_sample()`: difficult cases first (the five record checks, market-wide day, extremes, same-day filings, reference-only risk sections, broken summaries), then a random fill with seed 576. Text columns are cut to 300 characters, and the full lengths are kept in `*_chars` columns.

In [37]:
samples = {t: pd.read_csv(f"sample/{t}_sample.csv") for t in ["price_context", "news", "filings"]}
print({t: len(df) for t, df in samples.items()}, "total", sum(len(df) for df in samples.values()))
samples["price_context"].sample_reason.value_counts()

{'price_context': 40, 'news': 5, 'filings': 5} total 50


sample_reason
all 8 flagged (market-wide)          8
smallest range for symbol            8
filing same day                      7
largest range for symbol             5
random (seed 576)                    5
latest 10-Q risk refers to 10-K      3
record check 1: largest range        1
record check 2: first trading day    1
record check 3: filing same day      1
flagged day with low-info news       1
Name: count, dtype: int64

### 4.2 Output fingerprints (checklist #13)
SHA-256 of every output. A restart-and-run-all from the raw files must reproduce these exactly. The values are listed in the README.

In [38]:
outputs = sorted(Path("data/cleaned").glob("*.parquet")) + [Path("outputs/cleaning_changes.csv")] + sorted(Path("sample").glob("*.csv"))
pd.DataFrame({"file": [str(p) for p in outputs],
              "sha256_first16": [hashlib.sha256(p.read_bytes()).hexdigest()[:16] for p in outputs]})

,file,sha256_first16
0,data/cleaned/filings.parquet,0f2d1ca5868011c4
1,data/cleaned/news.parquet,7cf7dfeee7289326
2,data/cleaned/price_context.parquet,872060d0ae73cafa
3,data/cleaned/prices.parquet,b463e96a262bf2a7
4,outputs/cleaning_changes.csv,efa4f4169d1d6c2f
5,sample/filings_sample.csv,0ecd02f892e7aef0
6,sample/news_sample.csv,dada9f8dede493ec
7,sample/price_context_sample.csv,c54148844fc2747c
